<a href="https://colab.research.google.com/github/kokila83/AgenticAIMani_SampleMiniProject/blob/main/ReAct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Ex 1: Basic ReAct Pattern
import getpass
from openai import OpenAI

# 1. Get Azure AI Foundry API key securely
api_key = getpass.getpass("Enter your Azure AI Foundry API key: ")

# 2. Configure OpenAI client using your Azure AI Foundry endpoint
client = OpenAI(
    api_key=api_key.strip(),
    base_url="https://ganapathy-0333-resource.services.ai.azure.com/openai/v1/"
)

# 3. Define ReAct step function
def react_step(goal, history):
    prompt = f"""You are a ReAct agent. Follow this pattern:
Thought: [Your reasoning]
Action: [search/calculate/answer]
Action Input: [Input for the action]

Goal: {goal}
Previous steps: {history}

What's your next step?"""

    # Call Azure AI Foundry model deployment
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# 4. Test run
goal = "Find out what 25 * 48 is"
history = []

step1 = react_step(goal, history)
print(step1)

Enter your Azure AI Foundry API key: ··········
Thought: To find the product of 25 and 48, I need to multiply these two numbers.
Action: calculate
Action Input: 25 * 48


In [2]:
#Ex 2: Complete ReAct Agent
import getpass
import json
from openai import OpenAI

# 1. Get Azure AI Foundry API key securely
api_key = getpass.getpass("Enter your Azure AI Foundry API key: ")

# 2. Configure OpenAI client using your Azure AI Foundry base URL
client = OpenAI(
    api_key=api_key.strip(),
    base_url="https://ganapathy-0333-resource.services.ai.azure.com/openai/v1/"
)

CHAT_MODEL = "gpt-4.1-mini"

# 3. Define local tool functions
def search(query):
    data = {
        "capital of france": "Paris is the capital of France",
        "population of india": "India has 1.4 billion people",
        "ceo of google": "Sundar Pichai is CEO of Google"
    }
    for key, value in data.items():
        if key in query.lower():
            return value
    return f"No results for: {query}"

def calculate(expr):
    try:
        # Evaluate math expression safely
        return str(eval(expr))
    except Exception as e:
        return f"Error: {e}"

tools = {"search": search, "calculate": calculate}

# 4. ReAct Agent Function
def react_agent(goal, max_steps=5):
    history = []

    for step in range(max_steps):
        prompt = f"""Goal: {goal}
History: {json.dumps(history)}

Actions available:
- search(query)
- calculate(expression)
- answer(final_response)

Follow this exact output structure:
Thought: [Your step-by-step reasoning]
Action: [action name]
Action Input: [input value]"""

        # Call Azure AI Foundry model
        response = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=[{"role": "user", "content": prompt}]
        )

        content = response.choices[0].message.content.strip()
        print(f"\n--- Step {step + 1} ---\n{content}")

        lines = content.split('\n')
        action = None
        action_input = None

        for line in lines:
            if line.startswith("Action:"):
                action = line.split(":")[1].strip().lower()
            elif line.startswith("Action Input:"):
                action_input = line.split(":", 1)[1].strip()

        if action == "answer":
            print(f"\n✅ Final: {action_input}")
            return action_input
        elif action in tools:
            result = tools[action](action_input)
            history.append({"action": action, "input": action_input, "observation": result})
            print(f"Observation: {result}")
        else:
            print(f"Observation: Unknown action '{action}'. Retry with search, calculate, or answer.")

# 5. Execute Test Goal
if __name__ == "__main__":
    react_agent("What is the population of India divided by 100 million?")

Enter your Azure AI Foundry API key: ··········

--- Step 1 ---
Thought: To find the population of India divided by 100 million, I first need to find the current population of India. Since there is no prior information, I will search for the latest population of India.
Action: search
Action Input: current population of India
Observation: India has 1.4 billion people

--- Step 2 ---
Thought: The current population of India is 1.4 billion people. I need to divide this population by 100 million. Since 1 billion = 1000 million, 1.4 billion = 1400 million. Dividing 1400 million by 100 million equals 14.

Action: calculate
Action Input: 1.4e9 / 1e8
Observation: 14.0

--- Step 3 ---
Thought: The current population of India is given as 1.4 billion people. To find the population divided by 100 million, I convert 1.4 billion to numerical form (1.4e9) and divide it by 1e8. The calculation 1.4e9 / 1e8 equals 14.0, which means India's population divided by 100 million is 14.0.
Action: answer
Action

In [3]:
#Ex 3: Production ReAct Framework
import getpass
import json
from openai import OpenAI

# 1. Get Azure AI Foundry API key securely
api_key = getpass.getpass("Enter your Azure AI Foundry API key: ")

# 2. Configure OpenAI client using your Azure AI Foundry base URL
client = OpenAI(
    api_key=api_key.strip(),
    base_url="https://ganapathy-0333-resource.services.ai.azure.com/openai/v1/"
)

CHAT_MODEL = "gpt-4.1-mini"

# 3. Define local tool functions
def search(query):
    data = {
        "capital of france": "Paris is the capital of France",
        "population of india": "India has 1.4 billion people",
        "ceo of google": "Sundar Pichai is CEO of Google"
    }
    for key, value in data.items():
        if key in query.lower():
            return value
    return f"No results for: {query}"

def calculate(expr):
    try:
        # Evaluate math expression safely
        return str(eval(expr))
    except Exception as e:
        return f"Error: {e}"

tools = {"search": search, "calculate": calculate}

# 4. ReAct Agent Function
def react_agent(goal, max_steps=5):
    history = []

    for step in range(max_steps):
        prompt = f"""Goal: {goal}
History: {json.dumps(history)}

Actions available:
- search(query)
- calculate(expression)
- answer(final_response)

Follow this exact output structure:
Thought: [Your step-by-step reasoning]
Action: [action name]
Action Input: [input value]"""

        # Call Azure AI Foundry model
        response = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=[{"role": "user", "content": prompt}]
        )

        content = response.choices[0].message.content.strip()
        print(f"\n--- Step {step + 1} ---\n{content}")

        lines = content.split('\n')
        action = None
        action_input = None

        for line in lines:
            if line.startswith("Action:"):
                action = line.split(":")[1].strip().lower()
            elif line.startswith("Action Input:"):
                action_input = line.split(":", 1)[1].strip()

        if action == "answer":
            print(f"\n✅ Final: {action_input}")
            return action_input
        elif action in tools:
            result = tools[action](action_input)
            history.append({"action": action, "input": action_input, "observation": result})
            print(f"Observation: {result}")
        else:
            print(f"Observation: Unknown action '{action}'. Retry with search, calculate, or answer.")

# 5. Execute Test Goal
if __name__ == "__main__":
    react_agent("What is the population of India divided by 100 million?")

Enter your Azure AI Foundry API key: ··········

--- Step 1 ---
Thought: To find the population of India divided by 100 million, I first need to know the current population of India. I will search for the latest population data of India.
Action: search
Action Input: population of India 2024
Observation: India has 1.4 billion people

--- Step 2 ---
Thought: The population of India is 1.4 billion people. To find the population divided by 100 million, I need to calculate 1.4 billion ÷ 100 million. 

1.4 billion = 1,400,000,000  
100 million = 100,000,000

So the calculation is 1,400,000,000 / 100,000,000 = 14.

Action: calculate  
Action Input: 1400000000 / 100000000
Observation: 14.0

--- Step 3 ---
Thought: The population of India in 2024 is 1.4 billion people. To find the population divided by 100 million, I divide 1.4 billion (1,400,000,000) by 100 million (100,000,000), which equals 14.0.  
Action: answer  
Action Input: 14.0

✅ Final: 14.0
